# Research: Pairs trading a changement de regime (S&P 500)

**Serie** : `QuantConnect/research` — carnet de recherche local (yfinance), verdict honnete net de couts.

**Source** : article QC Research 21678 — *Regime-Switching Pairs Trading in the S&P 500* (D. Melchin), implementation de Yang, Tsai, Shyu & Chang (2016), *Int. Review of Economics and Finance* 43, 139-150 (doi 10.1016/j.iref.2015.10.036).

## Navigation

- [Index de la serie research](README.md)
- soeurs : [reversal a court terme](research_st_reversal.ipynb) · [plus-haut 52 semaines](research_fifty_two_week_high.ipynb) · [volatilite idiosyncratique](research_idio_vol_anomaly.ipynb)

> Carnet de recherche : le verdict est **falsifiable** et honnete — `BEATS` / `NO BEATS` / `INCONCLUSIVE`, jamais "prometteur".

## Objectifs d'apprentissage

1. Selectionner des **paires** d'actions par similarite de trajectoire (SSD, Gatev, Goetzmann & Rouwenhorst 2006), sans cointegration.
2. Estimer un **modele a changement de regime** du spread : deux regimes de regression a la moyenne (Vasicek) gouvernes par une chaine de Markov, via un **filtre de Hamilton** (estimation par EM).
3. Construire le **signal** : l'ecart du spread a sa **prevision conditionnelle au regime** le plus probable — et non le z-score du spread brut.
4. **Confronter honnetement** le modele a regime a une baseline de regression a la moyenne **constante** (le z-score classique), avec un decoupage in-sample / out-of-sample du P&L (estimation sur l'echantillon complet — cf. §5), net de couts, a paires et couts egaux.

## Hypothese

**H** : sur un univers de grandes capitalisations liquides du S&P 500 (sans Mag7), un modele de spread a **2 regimes** (chacun avec sa propre vitesse de reversion, son niveau et sa volatilite) genere un edge **superieur** a un modele a regime **unique** (z-score classique), a paires et couts egaux — parce que la bande d'entree et la prevision s'adaptent au regime actif du spread.

Un resultat `NO BEATS` (le modele a regime ne bat pas la baseline) est un resultat **valide** : il documente honnetement que la complexite additionnelle du changement de regime ne paie pas, net de couts, sur cet univers.

## Prequis

- `yfinance`, `pandas`, `numpy`, `scipy` (env de la serie).
- Notions : regression a la moyenne, modele autoregressif AR(1) (Vasicek), chaine de Markov, filtre de Hamilton (probabilites de regime filtreees), EM, backtest long-short, test de Diebold-Mariano.

In [1]:
"""Imports, configuration de l'univers (grandes caps S&P 500 liquides, sans Mag7) et chargement des prix."""
import numpy as np
import pandas as pd
from scipy import stats
import yfinance as yf

# Univers : grandes capitalisations liquides du S&P 500, SANS Mag7 (criteres C / anti-derive).
UNIVERSE = [
    "JPM", "BAC", "WFC", "BLK", "SCHW", "AXP", "MS", "BRK-B",
    "UNH", "LLY", "ABBV", "MRK", "JNJ", "PFE", "TMO", "ABT", "AMGN", "GILD",
    "CAT", "BA", "HON", "CSCO", "IBM", "LMT", "NEE", "DE",
    "ORCL", "ADBE", "CRM", "CTSH", "TXN", "QCOM", "MU", "AMAT", "INTC",
    "PG", "KO", "PEP", "MCD", "COST", "WMT", "CMG", "SBUX",
    "XOM", "CVX", "COP", "SLB",
    "LIN", "APD", "SHW", "FCX",
    "VZ", "T", "SO", "DUK",
]
UNIVERSE = sorted(set(UNIVERSE))

START = "2021-01-01"
END   = "2026-06-30"
IS_END = "2023-12-31"        # fin de la phase in-sample (estimation)
COST_BPS = 10                # 10 bps par reequilibrage
SSD_WINDOW = 252             # fenetre de similarite (1 an)
N_PAIRS = 5                  # nombre de paires retenues
SIGMA_ENTRY = 2.0            # seuil d'entree (en sigma du regime)
print(f"Univers declare : {len(UNIVERSE)} titres | {START} -> {END} | IS jusqu'a {IS_END} | {N_PAIRS} paires | seuil {SIGMA_ENTRY} sigma")

prices = yf.download(UNIVERSE, start=START, end=END, auto_adjust=True, progress=False)["Close"]
prices = prices.dropna(axis=1, how="any")
N_UNIV = prices.shape[1]
print(f"Prix reels charges : {N_UNIV} titres x {prices.shape[0]} jours")

Univers declare : 55 titres | 2021-01-01 -> 2026-06-30 | IS jusqu'a 2023-12-31 | 5 paires | seuil 2.0 sigma


<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:531: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:531: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:531: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_ind

<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:531: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:547: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:547: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.so

<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:547: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:476: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:476: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
<USER_PATH>\.conda\envs\coursia-canoni

<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:476: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:480: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:476: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packag

<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:531: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:545: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:547: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  divide

... (3 outputs left out)

<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:476: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:480: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:476: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packag

<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:531: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:531: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:545: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_in

Prix reels charges : 55 titres x 1377 jours


<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:480: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:545: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
<USER_PATH>\.conda\envs\coursia-canonical\Lib\site-packages\yfinance\utils.py:476: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
<USER_PATH>\.conda\

## 1. Selection des paires par SSD

Le **SSD** (sum of squared differences) mesure la similarite de trajectoire de deux actions : on rebase les prix de chaque action a 1 au debut de la fenetre, puis on somme les differences carrees sur la fenetre. Plus le SSD est **faible**, plus les deux trajectoires sont proches — c'est le critere de selection de Gatev, Goetzmann & Rouwenhorst (2006), sans cointegration.

In [2]:
"""Selection des N paires les plus similaires par SSD (Gatev 2006), sur la fenetre initiale."""
def ssd_select(log_prices: pd.DataFrame, window: int, n_pairs: int):
    seg = log_prices.iloc[:window]
    norm = (seg / seg.iloc[0]).to_numpy()            # rebase a 1 en debut de fenetre
    d = norm
    diff = d[:, :, None] - d[:, None, :]            # [t, i, j] = norm[t, i] - norm[t, j]
    ssd = (diff ** 2).sum(axis=0)                    # [i, j]
    cols = list(log_prices.columns)
    cands = [(float(ssd[i, j]), cols[i], cols[j])
             for i in range(len(cols)) for j in range(i + 1, len(cols))]
    cands.sort()
    return [(a, b) for _, a, b in cands[:n_pairs]]

logp = np.log(prices)
PAIRS = ssd_select(logp, SSD_WINDOW, N_PAIRS)
print(f"{N_PAIRS} paires retenues (SSD le plus faible, fenetre {SSD_WINDOW} j) :")
for k, (a, b) in enumerate(PAIRS):
    print(f"  paire {k + 1} : {a} / {b}")

5 paires retenues (SSD le plus faible, fenetre 252 j) :
  paire 1 : LIN / UNH
  paire 2 : LIN / MCD
  paire 3 : BLK / DUK
  paire 4 : BLK / LIN
  paire 5 : SHW / UNH


## 2. Le spread et les deux modeles de reversion

Le **spread** d'une paire est la difference des log-prix : `s_t = ln P_A - ln P_B`. Un spread stationnaire (reversible a la moyenne) est la condition d'un pairs trading.

On confronte **deux modeles** sur le meme spread :
- **Baseline (regime unique)** : un AR(1) de Vasicek a parametres constants — le z-score classique.
- **Modele a regime (la contribution de l'article)** : deux regimes de Vasicek `{kappa_j, theta_j, sigma_j}` (j = 1, 2) gouvernes par une chaine de Markov ; le **filtre de Hamilton** donne, a chaque date, la probabilite d'etre dans chaque regime.

In [3]:
"""Modele baseline : AR(1) de Vasicek a regime unique (z-score classique).

ds = kappa*(theta - s_{t-1}) + e  soit  ds = (kappa*theta) - kappa*s_{t-1} + e,
regression de ds sur [1, s_{t-1}].
"""
def fit_ar1(s: np.ndarray) -> dict:
    ds = np.diff(s)
    s_prev = s[:-1]
    X = np.column_stack([np.ones_like(s_prev), -s_prev])
    beta, *_ = np.linalg.lstsq(X, ds, rcond=None)
    kappa = float(-beta[1])
    theta = float(beta[0] / kappa) if abs(kappa) > 1e-9 else 0.0
    resid = ds - (beta[0] + beta[1] * s_prev)
    return {"kappa": kappa, "theta": theta, "sigma": float(np.std(resid))}

def ar1_forecast(s_prev: float, p: dict) -> float:
    return s_prev + p["kappa"] * (p["theta"] - s_prev)

_s = np.log(prices[PAIRS[0][0]]) - np.log(prices[PAIRS[0][1]])
_p = fit_ar1(_s.values)
print(f"Baseline AR(1), paire 1 ({PAIRS[0][0]}/{PAIRS[0][1]}) : "
      f"kappa={_p['kappa']:.4f}  theta={_p['theta']:.4f}  sigma={_p['sigma']:.4f}")

Baseline AR(1), paire 1 (LIN/UNH) : kappa=-0.0020  theta=-0.0245  sigma=0.0214


## 3. Le modele a 2 regimes + filtre de Hamilton

C'est la **contribution distinctive** de l'article. Le spread suit :

    ds_t = kappa_{S_t} (theta_{S_t} - s_{t-1}) + eps_t ,   eps_t ~ N(0, sigma_{S_t}^2)

ou `S_t in {1, 2}` est une chaine de Markov (probabilites de transition `p11`, `p22`). Le **filtre de Hamilton** calcule, a chaque date `t`, la probabilite filtreee d'etre dans chaque regime `xi_t = P(S_t=j | s_{<=t})`. L'estimation des 8 parametres se fait par **EM** : l'etape E est le filtre, l'etape M re-estime les parametres de chaque regime, ponderees par `xi_t`.

In [4]:
"""Modele a 2 regimes de Vasicek + filtre de Hamilton, estime par EM.

8 parametres : par regime j -> kappa_j, theta_j, sigma_j ; + transitions p11, p22.
Le filtre est en espace log (stabilite). E = filtre, M = re-estimation ponderee par xi.
"""
def _regime_loglik(s_prev: float, s_t: float, j: int, par: dict) -> float:
    k, th, sg = par["kappa"][j], par["theta"][j], par["sigma"][j]
    m = s_prev + k * (th - s_prev)
    return -0.5 * ((s_t - m) / sg) ** 2 - np.log(sg)

def hamilton_filter(s, par, p11, p22):
    T = len(s)
    xi = np.zeros((T, 2))
    log_xi = np.zeros((T, 2))
    for t in range(1, T):
        s_prev, s_t = s[t - 1], s[t]
        log_pred = np.log([p11, 1.0 - p22]) + log_xi[t - 1]
        log_pred -= np.log(np.exp(log_pred - log_pred.max()).sum())
        ll = np.array([_regime_loglik(s_prev, s_t, 0, par),
                       _regime_loglik(s_prev, s_t, 1, par)])
        log_xi[t] = log_pred + ll
        log_xi[t] -= np.log(np.exp(log_xi[t] - log_xi[t].max()).sum())
        xi[t] = np.exp(log_xi[t] - log_xi[t].max())
        xi[t] /= xi[t].sum()
    xi[0] = [0.5, 0.5]
    return xi

def em_regime(s, n_iter=40, tol=1e-6):
    T = len(s)
    ds = np.diff(s); s_prev = s[:-1]
    base = float(np.std(ds)) + 1e-6
    par = {"kappa": [0.05, 0.15], "theta": [0.0, 0.0], "sigma": [base, base]}
    p11, p22 = 0.98, 0.98
    prev = None
    for _ in range(n_iter):
        xi = hamilton_filter(s, par, p11, p22)
        w = xi[:-1]
        new = {"kappa": [], "theta": [], "sigma": []}
        for j in range(2):
            X = np.column_stack([np.ones(T - 1), -s_prev])
            XTW = X.T * w[:, j]
            beta, *_ = np.linalg.lstsq(XTW @ X, XTW @ ds, rcond=None)
            k = float(-beta[1]); th = float(beta[0] / k) if abs(k) > 1e-9 else 0.0
            resid = ds - (beta[0] + beta[1] * s_prev)
            sg = float(np.sqrt((w[:, j] * resid ** 2).sum() / max(w[:, j].sum(), 1e-9))) + 1e-9
            new["kappa"].append(k); new["theta"].append(th); new["sigma"].append(sg)
        p11 = float(np.clip((w[:, 0] * xi[1:, 0]).sum() / max(w[:, 0].sum(), 1e-9), 0.5, 0.999))
        p22 = float(np.clip((w[:, 1] * xi[1:, 1]).sum() / max(w[:, 1].sum(), 1e-9), 0.5, 0.999))
        par = new
        cur = (p11, p22) + tuple(par["kappa"]) + tuple(par["sigma"])
        if prev is not None and abs(sum(c - p for c, p in zip(cur, prev))) < tol:
            break
        prev = cur
    xi = hamilton_filter(s, par, p11, p22)
    return par, p11, p22, xi

# demonstration sur la paire 1
_s = np.log(prices[PAIRS[0][0]]) - np.log(prices[PAIRS[0][1]])
par1, p11, p22, xi1 = em_regime(_s.values)
share1 = xi1.mean(axis=0)
print(f"Modele a regime, paire 1 ({PAIRS[0][0]}/{PAIRS[0][1]}) :")
print(f"  regime 1 : kappa={par1['kappa'][0]:.3f} theta={par1['theta'][0]:.3f} sigma={par1['sigma'][0]:.3f}")
print(f"  regime 2 : kappa={par1['kappa'][1]:.3f} theta={par1['theta'][1]:.3f} sigma={par1['sigma'][1]:.3f}")
print(f"  transitions : p11={p11:.3f} p22={p22:.3f} | part du temps : regime1={share1[0]:.2f} regime2={share1[1]:.2f}")

Modele a regime, paire 1 (LIN/UNH) :
  regime 1 : kappa=-0.002 theta=-0.021 sigma=0.021
  regime 2 : kappa=-1.308 theta=0.256 sigma=0.725
  transitions : p11=0.999 p22=0.500 | part du temps : regime1=1.00 regime2=0.00


## 4. Le signal : prevision conditionnelle au regime

La difference cruciale avec le z-score classique : la bande d'entree n'est **pas** le z-score du spread brut, mais l'ecart du spread a sa **prevision conditionnelle au regime le plus probable**, mesuree en **sigma de ce regime** :

- `F_t` = prevision 1-pas du regime le plus probable (son kappa, son theta)
- entree si `s_t - F_t >= +2*sigma` (on va court du spread) ou `<= -2*sigma` (long du spread)
- le signal est **recalcule chaque jour** : le retour du spread a sa prevision referme la position (sortie inherente, sans etat a maintenir)
- le regime est lu a `t-1` : **aucun lookahead**

(Le stop temporel par demi-vie du regime actif, present dans l'article, est un exercice — section 8.)

In [5]:
"""Position {-1, 0, +1} par spread, selon le modele (stateless : recalcule chaque jour)."""
def spread_position(s: np.ndarray, mode: str, sigma_entry: float = SIGMA_ENTRY) -> np.ndarray:
    s = np.asarray(s, dtype=float)
    if mode == "ar1":
        par = fit_ar1(s)
        F = np.array([ar1_forecast(s[i - 1], par) for i in range(len(s))])
        z = (s - F) / (par["sigma"] + 1e-9)
        return np.where(z <= -sigma_entry, 1, np.where(z >= sigma_entry, -1, 0)).astype(int)
    par, p11, p22, xi = em_regime(s)
    k_active = np.where(xi[:, 0] >= xi[:, 1], 0, 1)   # regime le plus probable
    sig = np.zeros(len(s), dtype=int)
    for i in range(1, len(s)):
        j = k_active[i - 1]                            # regime connu a t-1
        F = s[i - 1] + par["kappa"][j] * (par["theta"][j] - s[i - 1])
        dev = (s[i] - F) / (par["sigma"][j] + 1e-9)
        sig[i] = 1 if dev <= -sigma_entry else (-1 if dev >= sigma_entry else 0)
    return sig

print("Fonctions de signal definies : fit_ar1, ar1_forecast, em_regime, spread_position (2 modes)")

Fonctions de signal definies : fit_ar1, ar1_forecast, em_regime, spread_position (2 modes)


## 5. Backtest net de couts, decoupage in-sample / out-of-sample du P&L

Les deux modeles (a regime et baseline) sont estimes sur l'**echantillon complet** (2021-01 → 2026-06) : `evaluate()` passe la serie integre a `spread_position`, qui y estime les parametres (OLS pour la baseline, EM + filtre de Hamilton pour le regime). Le decoupage **in-sample / out-of-sample** s'applique ensuite au **P&L** a `IS_END`, **pas** a l'estimation. C'est donc une comparaison **a echantillon complet**, decoupee pour le rapport — et **non** un test out-of-sample au sens ou les parametres ne verraient pas les dates de la jambe de test. La position est decidee a `t-1` et executee sur le retour de spread `ds_t` ; les couts sont charges au turnover.

**Limite honnete :** un vrai test hors echantillon exigerait de geler les parametres estimes sur l'in-sample et de les appliquer sur l'out-of-sample (ou un walk-forward). Ce carnet ne le fait pas : la jambe « out-of-sample » est un decoupage du P&L, pas une generalisation sur des dates non vues.

In [6]:
"""Moteur de backtest + confrontation des deux modeles (estimation IS, evaluation OOS)."""
def backtest_spread(ds: np.ndarray, position: np.ndarray, cost_bps: float) -> np.ndarray:
    pos = np.zeros(len(position), dtype=float)
    pos[1:] = position[:-1]                            # position a t-1, executee sur ds_t
    gross = pos * ds
    turnover = np.abs(np.diff(position, prepend=0.0))
    cost = turnover * cost_bps / 1e4
    pnl = gross - cost
    pnl[0] = 0.0
    return pnl

def evaluate(pairs, mode, sigma_entry=SIGMA_ENTRY):
    is_parts, oos_parts = [], []
    for a, b in pairs:
        s = (np.log(prices[a]) - np.log(prices[b])).dropna()
        ds = s.diff().fillna(0.0).values
        pos = spread_position(s.values, mode, sigma_entry)
        pnl = backtest_spread(ds, pos, COST_BPS)
        m = (s.index <= pd.Timestamp(IS_END))
        is_parts.append(pd.Series(pnl[m])); oos_parts.append(pd.Series(pnl[~m]))
    return (pd.concat(is_parts, axis=1).mean(axis=1),
            pd.concat(oos_parts, axis=1).mean(axis=1))

def perf(pnl: pd.Series):
    r = pnl.dropna()
    ann = float(r.mean()) * 252
    vol = float(r.std(ddof=1)) * np.sqrt(252)
    return ann, (ann / vol if vol > 1e-9 else 0.0)

ar1_is, ar1_oos = evaluate(PAIRS, "ar1")
reg_is, reg_oos = evaluate(PAIRS, "regime")
print("=== IN-SAMPLE (estimation) ===")
for name, p in [("baseline AR(1)", ar1_is), ("modele a regime", reg_is)]:
    a, s = perf(p); print(f"  {name:16s} : CAGR {a*100:6.2f}%   Sharpe {s:5.2f}")
print("=== OUT-OF-SAMPLE (evaluation) ===")
for name, p in [("baseline AR(1)", ar1_oos), ("modele a regime", reg_oos)]:
    a, s = perf(p); print(f"  {name:16s} : CAGR {a*100:6.2f}%   Sharpe {s:5.2f}")

=== IN-SAMPLE (estimation) ===
  baseline AR(1)   : CAGR  -3.62%   Sharpe -0.60
  modele a regime  : CAGR  -0.20%   Sharpe -0.06
=== OUT-OF-SAMPLE (evaluation) ===
  baseline AR(1)   : CAGR  -3.74%   Sharpe -0.51
  modele a regime  : CAGR  -0.85%   Sharpe -0.18


## 6. Test de Diebold-Mariano (OOS, net de couts)

On teste si le modele a regime **sur-performe significativement** la baseline OOS : `d = baseline - regime` (positif = la baseline gagne). Le test est **unilateral** : le modele a regime "gagne" seulement si `d` est significativement **négative** (stat < 0, p < 0.05).

In [7]:
"""Test de Diebold-Mariano unilateral sur d = baseline - regime (OOS, variance HAC)."""
def dm_unilateral(d: np.ndarray, maxlag: int = 5):
    d = np.asarray(d, dtype=float); n = len(d)
    dbar = d.mean()
    var = (d ** 2).mean()
    for k in range(1, maxlag + 1):
        var += 2.0 * (1.0 - k / (maxlag + 1)) * (d[k:] * d[:-k]).mean()
    var /= n
    if var <= 0:
        return 0.0, 1.0
    stat = dbar / np.sqrt(var)
    p_one = float(1.0 - stats.norm.cdf(stat))
    return float(stat), p_one

d = (ar1_oos - reg_oos).dropna().values
stat, p_one = dm_unilateral(d)
reg_wins = (stat < 0) and (p_one < 0.05)
print(f"DM OOS (d = baseline - regime) : stat = {stat:.3f} | p unilaterale = {p_one:.4f}")
print(f"Le modele a regime bat la baseline OOS : {reg_wins}")

DM OOS (d = baseline - regime) : stat = -0.852 | p unilaterale = 0.8028
Le modele a regime bat la baseline OOS : False


## 7. Robustesse (sensibilite au seuil) et verdict

L'article montre que la **largeur de la bande** est sensible au seuil en sigma. On re-parcourt `sigma_entry` sur `{1.8, 2.0, 2.2}` et releve le delta OOS (regime - baseline) : un edge reel doit etre **stable** dans le signe, pas un pic isole. Le verdict croise le signe du delta OOS et le test DM unilateral.

In [8]:
"""Robustesse : delta OOS (regime - baseline) selon le seuil, puis verdict honnete."""
print("Seuil (sigma)   baseline OOS    regime OOS    delta (regime - baseline)")
for se in [1.8, 2.0, 2.2]:
    b_oos = evaluate(PAIRS, "ar1", se)[1]
    r_oos = evaluate(PAIRS, "regime", se)[1]
    ba, _ = perf(b_oos); ra, _ = perf(r_oos)
    print(f"  {se:9.1f}     {ba*100:8.2f}%      {ra*100:8.2f}%      {(ra - ba)*100:+8.2f}%")

delta_oos = float((reg_oos - ar1_oos).dropna().mean())
if reg_wins and delta_oos > 0:
    verdict = "BEATS"
elif delta_oos < 0:
    verdict = "NO BEATS"
else:
    verdict = "INCONCLUSIVE"
print(f"\nDelta OOS moyen (regime - baseline) : {delta_oos*100:+.4f}%/jour")
print(f"DM unilateral en faveur du regime   : {reg_wins}")
print(f"VERDICT : {verdict}  (modele a regime vs baseline z-score, OOS, net de {COST_BPS} bps, {N_PAIRS} paires)")

Seuil (sigma)   baseline OOS    regime OOS    delta (regime - baseline)


        1.8        -3.77%         -1.33%         +2.44%


        2.0        -3.74%         -0.85%         +2.88%


        2.2        -4.81%         -0.10%         +4.71%

Delta OOS moyen (regime - baseline) : +0.0114%/jour
DM unilateral en faveur du regime   : False
VERDICT : INCONCLUSIVE  (modele a regime vs baseline z-score, OOS, net de 10 bps, 5 paires)


## 8. Exercices d'extension

### Exercice 1 -- Selection par SSD sur une fenetre glissante

Re-ecrit `ssd_select` pour recalculer les paires **chaque mois** sur une fenetre glissante de 252 jours (au lieu d'une selection unique sur la fenetre initiale), et retourner la liste des paires par mois.

In [9]:
# TODO etudiant : reecrir ssd_select en version glissante (re-selection mensuelle)
# def ssd_select_rolling(log_prices, window=252, n_pairs=5): ...
# monthly_pairs = None  # TODO etudiant : dict {mois: [(a, b), ...]}
print("Exercice a completer")

Exercice a completer


### Exercice 2 -- Ajouter un 3e regime

Etendre `em_regime` a **3 regimes** (3 jeux de kappa/theta/sigma + matrice de transition 3x3) et relever si le 3e regime est effectivement sollicite par le filtre (part du temps passe dans chaque regime).

In [10]:
# TODO etudiant : generaliser em_regime a 3 regimes ; renvoyer la part du temps par regime
# regime_share = None  # TODO etudiant : dict {regime: fraction du temps}
print("Exercice a completer")

Exercice a completer


### Exercice 3 -- Le stop temporel par demi-vie

Implémenter le **stop temporel** de l'article : borner la detention de chaque position a la demi-vie du regime actif `h = ln(0.5)/ln|1-kappa|` (re-remettre a plat au-dela), et mesurer l'effet sur le P&L OOS net de couts.

In [11]:
# TODO etudiant : ajouter le stop temporel par demi-vie dans spread_position
# result_ex3 = None  # TODO etudiant : P&L OOS avec / sans le stop
print("Exercice a completer")

Exercice a completer


## Conclusion

Ce carnet replique, sur un univers de grandes capitalisations du S&P 500 (sans Mag7), la contribution distinctive de l'article 21678 : un **modele de spread a changement de regime** (2 regimes de Vasicek + filtre de Hamilton, estimation par EM) dont le signal est l'ecart a la **prevision conditionnelle au regime**, confronte honnetement a une baseline de z-score classique, avec decoupage in-sample / out-of-sample du P&L et net de couts (estimation sur l'echantillon complet — cf. §5). Le verdict est **falsifiable** : il dit si la complexite additionnelle du changement de regime paie reellement, ou si elle se perd dans les couts.

Deux points methodologiques sont porteurs : (1) le signal est **stateless** — recalcule chaque jour, le retour du spread a sa prevision referme la position naturellement, sans etat a maintenir ; (2) la comparaison est **a paires et couts egaux** — la seule difference entre les deux branches est le modele de prevision, donc le verdict isole precisement la valeur ajoutee du regime.